In [1]:
import pandas as pd
import re
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Tải dữ liệu ngôn ngữ
nltk.download('stopwords', quiet=True)
tqdm.pandas(desc="Tiến độ làm sạch LSA")

# Sử dụng các đường dẫn đã thống nhất từ file base
RAW_DIR = "data/raw"
PREPROCESSED_DIR = "data/processed"

In [2]:
# Đọc file đã tải từ bước base
df_corpus = pd.read_parquet(f"../{RAW_DIR}/corpus.parquet")

# Xử lý các giá trị trống
df_corpus['title'] = df_corpus['title'].fillna('')
df_corpus['text'] = df_corpus['text'].fillna('')

print(f"Đã tải {df_corpus.shape[0]:,} bài báo.")

Đã tải 25,657 bài báo.


In [3]:
# Chỉ giữ stop_words (xóa PorterStemmer vì không dùng nữa)
stop_words = set(stopwords.words('english'))

In [4]:
# Preprocessing nhẹ hơn, phù hợp với văn bản khoa học (SciDocs)
def preprocess_lsa_scidocs(text):
    if not isinstance(text, str):
        return ""
    # Giữ chữ cái, số, dấu trừ, chấm - rất quan trọng với thuật ngữ chuyên môn
    text = re.sub(r'[^a-zA-Z0-9\s\-\.\,]', ' ', text.lower())
    tokens = [w for w in text.split() 
              if len(w) > 1 
              and w not in stop_words]
    return " ".join(tokens)

# Tăng mạnh trọng số tiêu đề (Title weighting)
df_corpus['raw_content'] = (df_corpus['title'] + " ") * 5 + df_corpus['text']

print("Bắt đầu quá trình xử lý văn bản cho LSA...")
df_corpus['text_lsa'] = df_corpus['raw_content'].progress_apply(preprocess_lsa_scidocs)

# Lọc bỏ dòng rỗng
# Lọc bỏ dòng rỗng và đổi tên cột text thành abstract để làm Web
df_tv2 = df_corpus[['_id', 'title', 'text', 'text_lsa']]\
            .rename(columns={'text': 'abstract'})\
            .dropna(subset=['text_lsa']).reset_index(drop=True)

# Lưu file
output_path = f"../{PREPROCESSED_DIR}/data_tv2.parquet"
df_tv2.to_parquet(output_path, index=False)
print(f"Đã lưu dữ liệu LSA tại: {output_path}")

Bắt đầu quá trình xử lý văn bản cho LSA...


Tiến độ làm sạch LSA:   0%|          | 0/25657 [00:00<?, ?it/s]

Đã lưu dữ liệu LSA tại: ../data/processed/data_tv2.parquet
